[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/22.%20The%20Internal%20Vehicle%20%28Terminal%20Truck%29%20Dispatching%20Problem/P22-Tier-4_executed.ipynb)

---



# 22. The Internal Vehicle (Terminal Truck) Dispatching Problem
## Tier 4: Self-Supervised Learning for Predictive Dispatching

### Goal

This notebook builds a self-supervised representation-learning example for terminal operations. It uses an LSTM with attention to encode short sequences of terminal states and a contrastive objective to bring representations of temporally adjacent sequences closer than representations of distant sequences. A policy network is included to illustrate how an action-scoring head can be connected to the learned representation.

The synthetic example trains the encoder and contrastive projection head. It does **not** provide dispatch labels or train the policy network. Consequently, the policy's assignments are untrained scores-based demonstrations, not evidence of a learned or operationally effective dispatch policy. The performance comparisons later in the notebook are descriptive evaluations of those demonstrations, not proof that predictive dispatching outperforms a reactive policy.

### Inputs, outputs, and scope

- **Inputs:** Synthetic snapshots of trucks and container jobs, including locations, availability, load, priority, and simple system-level indicators.
- **Representation-learning output:** A fixed-length vector for each short sequence of terminal states.
- **Dispatch demonstration output:** A one-to-one greedy assignment of some containers to trucks using action scores from the policy head.
- **Scope:** The example has at most five represented trucks and five represented containers per state. It does not model travel-time dynamics, equipment interference, capacity constraints, time-window feasibility, route planning, or a validated operational objective. The generated snapshots are independent random states rather than a calibrated terminal simulator.

### Modeling assumptions

Adjacent observations in the generated history are treated as positive pairs, and observations separated by more than ten periods are treated as candidate negative pairs. This is a sampling assumption for the contrastive task; temporal proximity does not guarantee operational similarity, and temporal distance does not guarantee operational dissimilarity. The model therefore learns the structure induced by this synthetic sampling rule, not a verified notion of operational state similarity.

A state is represented by five truck slots, five container slots, and three system-level features. Each truck contributes four features, each container contributes four features, and the system contributes three features, so the input dimension is

$$
5(4) + 5(4) + 3 = 43.
$$

Missing slots are zero-padded. Container features are taken in the order in which containers appear in the state; no sorting or permutation-invariant aggregation is applied.

### Method overview

1. **Temporal encoding:** An LSTM processes a fixed-length window of state vectors. Multi-head self-attention combines information across the window, and mean pooling produces one sequence representation.
2. **Contrastive learning:** For an anchor, an adjacent positive, and a temporally distant negative, an InfoNCE-style cross-entropy loss encourages the anchor representation to be more similar to the positive than to the negative.
3. **Policy head:** A neural network maps an encoder representation to action logits. In this notebook, this head is not trained because the contrastive loss does not depend on its parameters.
4. **Demonstration and evaluation:** The untrained action scores are converted to one-to-one assignments and compared descriptively with a simple priority-based reactive baseline.

### Mathematical formulation

Let $x_t \in \mathbb{R}^{43}$ denote the feature vector for the state at period $t$. A window of length $L=5$ is $X_t=(x_{t-L+1},\ldots,x_t)$. The temporal encoder $f_\theta$ maps this window to an embedding $z_t=f_\theta(X_t)\in\mathbb{R}^{128}$, and the projection head $g_\phi$ maps it to a unit vector $q_t=g_\phi(z_t)/\|g_\phi(z_t)\|_2$.

For an anchor window $X_a$, a positive window $X_p$, and a negative window $X_n$, the notebook minimizes

$$
\ell(a,p,n) = -\log\frac{\exp(q_a^\top q_p / \tau)}{\exp(q_a^\top q_p / \tau)+\exp(q_a^\top q_n / \tau)},
$$

where $\tau=0.1$ is the temperature. The batch objective is the mean of this loss over sampled triples. The anchor-positive and anchor-negative dot products are the two logits passed to cross-entropy, with the positive class assigned label zero. The trainable parameters in this objective are the encoder parameters $\theta$ and projection-head parameters $\phi$; the policy parameters do not appear in the loss.

For the synthetic history, positive windows are shifted by one period and negative windows end more than ten periods away from the anchor. The first four history entries cannot form a complete five-period window, so they are not used as anchors. The number of possible windows grows linearly with the history length; the encoder processes each sampled batch with work that grows with batch size and window length. Attention over a window of length $L$ has pairwise attention work proportional to $L^2$.

### What to look for

- Whether the contrastive training loss changes over epochs. A lower loss indicates easier separation of the sampled positive and negative pairs under this objective; it is not, by itself, a measure of representation quality or dispatch performance.
- How the assignment demonstration behaves under the fixed action-score layout and one-to-one greedy matching rule.
- Why a trained encoder does not imply a trained dispatch policy when the policy parameters receive no training signal.

The example uses 100 synthetic periods and fixed random seeds for reproducibility. Exact training time and learned values depend on the available hardware and software environment.

In [ ]:
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim


# Fixed seeds make the synthetic sampling and parameter initialization reproducible.
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Initializing self-supervised representation learning for terminal dispatching.")

In [ ]:
class TemporalEncoder(nn.Module):
    """Map a sequence of terminal-state vectors to one fixed-size embedding."""

    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int = 2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # The LSTM first builds a contextual representation at each time step.
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2 if num_layers > 1 else 0.0,
        )

        # Attention lets each time step use information from the other steps.
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=8,
            dropout=0.1,
            batch_first=True,
        )
        self.layer_norm = nn.LayerNorm(hidden_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Encode input shaped (batch_size, sequence_length, input_dim)."""
        lstm_out, _ = self.lstm(x)
        attended, _ = self.attention(lstm_out, lstm_out, lstm_out)

        # The residual connection retains LSTM information while adding attention.
        attended = self.layer_norm(lstm_out + attended)
        return torch.mean(attended, dim=1)


class ContrastiveLearner(nn.Module):
    """Project encoder embeddings and calculate a two-candidate InfoNCE loss."""

    def __init__(self, encoder_dim: int, projection_dim: int):
        super().__init__()
        self.projector = nn.Sequential(
            nn.Linear(encoder_dim, projection_dim),
            nn.ReLU(),
            nn.Linear(projection_dim, projection_dim),
        )

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        projections = self.projector(embeddings)
        return nn.functional.normalize(projections, p=2, dim=1)

    def contrastive_loss(
        self,
        anchor: torch.Tensor,
        positive: torch.Tensor,
        negative: torch.Tensor,
        temperature: float = 0.1,
    ) -> torch.Tensor:
        """Prefer the positive pair over the sampled negative pair."""
        anchor_proj = self(anchor)
        positive_proj = self(positive)
        negative_proj = self(negative)

        positive_similarity = torch.sum(anchor_proj * positive_proj, dim=1) / temperature
        negative_similarity = torch.sum(anchor_proj * negative_proj, dim=1) / temperature
        logits = torch.stack((positive_similarity, negative_similarity), dim=1)
        labels = torch.zeros(anchor.size(0), dtype=torch.long, device=anchor.device)
        return nn.functional.cross_entropy(logits, labels)


class DispatchingPolicy(nn.Module):
    """Produce action logits from an encoder representation."""

    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, action_dim),
        )

    def forward(self, state_representation: torch.Tensor) -> torch.Tensor:
        return self.network(state_representation)


print("Temporal encoder, contrastive learner, and policy head are defined.")

In [ ]:
class SelfSupervisedDispatcher:
    """Train a temporal encoder with contrastive pairs and expose an action head."""

    def __init__(self, terminal_config: Dict[str, int]):
        self.config = terminal_config
        self.max_trucks = terminal_config["num_trucks"]
        self.max_containers = terminal_config["max_containers"]
        self.window_length = terminal_config["window_length"]
        self.fixed_feature_size = terminal_config["state_dim"]

        expected_feature_size = 4 * self.max_trucks + 4 * self.max_containers + 3
        if self.fixed_feature_size != expected_feature_size:
            raise ValueError(
                f"state_dim must be {expected_feature_size} for the configured feature slots."
            )
        if terminal_config["num_actions"] < self.max_trucks * self.max_containers:
            raise ValueError("num_actions must cover every configured truck-container pair.")

        self.encoder = TemporalEncoder(
            input_dim=self.fixed_feature_size,
            hidden_dim=128,
            num_layers=2,
        ).to(device)
        self.contrastive_learner = ContrastiveLearner(
            encoder_dim=128,
            projection_dim=64,
        ).to(device)
        self.policy = DispatchingPolicy(
            state_dim=128,
            action_dim=terminal_config["num_actions"],
        ).to(device)

        # The contrastive objective trains the encoder and projector only.
        # The policy is retained as an untrained action-scoring demonstration.
        self.optimizer = optim.Adam(
            list(self.encoder.parameters())
            + list(self.contrastive_learner.parameters()),
            lr=0.001,
        )
        self.training_losses: List[float] = []
        self.loss_change_rates: List[float] = []

    def encode_terminal_state(self, state_dict: Dict) -> torch.Tensor:
        """Convert one state to the configured fixed-length feature vector."""
        features: List[float] = []

        for i in range(self.max_trucks):
            truck_id = f"T{i + 1}"
            truck = state_dict["trucks"].get(truck_id)
            if truck is None:
                features.extend([0.0, 0.0, 0.0, 0.0])
            else:
                features.extend(
                    [
                        truck["location_x"],
                        truck["location_y"],
                        truck["available_time"],
                        truck["current_load"],
                    ]
                )

        for i in range(self.max_containers):
            if i < len(state_dict["containers"]):
                container = state_dict["containers"][i]
                features.extend(
                    [
                        container["origin_x"],
                        container["origin_y"],
                        container["dest_x"],
                        container["dest_y"],
                    ]
                )
            else:
                features.extend([0.0, 0.0, 0.0, 0.0])

        features.extend(
            [
                len(state_dict["containers"]),
                state_dict["average_urgency"],
                state_dict["truck_utilization"],
            ]
        )
        if len(features) != self.fixed_feature_size:
            raise ValueError("Encoded state does not match the configured feature dimension.")
        return torch.tensor(features, dtype=torch.float32)

    def create_training_pairs(
        self, historical_data: List[Dict]
    ) -> List[Tuple[List[Dict], List[Dict], List[Dict]]]:
        """Create adjacent positive windows and temporally distant negatives."""
        pairs = []
        last_anchor = len(historical_data) - self.window_length

        for anchor_start in range(last_anchor):
            positive_start = anchor_start + 1
            anchor_window = historical_data[
                anchor_start : anchor_start + self.window_length
            ]
            positive_window = historical_data[
                positive_start : positive_start + self.window_length
            ]

            negative_starts = [
                start
                for start in range(last_anchor + 1)
                if abs(start - anchor_start) > 10
            ]
            if negative_starts:
                negative_start = random.choice(negative_starts)
                negative_window = historical_data[
                    negative_start : negative_start + self.window_length
                ]
                pairs.append((anchor_window, positive_window, negative_window))
        return pairs

    def train_epoch(self, training_data: List[Dict]) -> float:
        """Train one epoch on sampled temporal contrastive triples."""
        self.encoder.train()
        self.contrastive_learner.train()
        pairs = self.create_training_pairs(training_data)
        if not pairs:
            return 0.0

        epoch_loss = 0.0
        num_batches = 0
        batch_size = 16

        for start in range(0, len(pairs), batch_size):
            batch_pairs = pairs[start : start + batch_size]
            anchor_batch = []
            positive_batch = []
            negative_batch = []

            for anchor_window, positive_window, negative_window in batch_pairs:
                anchor_batch.append(
                    torch.stack(
                        [self.encode_terminal_state(state) for state in anchor_window]
                    )
                )
                positive_batch.append(
                    torch.stack(
                        [self.encode_terminal_state(state) for state in positive_window]
                    )
                )
                negative_batch.append(
                    torch.stack(
                        [self.encode_terminal_state(state) for state in negative_window]
                    )
                )

            anchor_tensor = torch.stack(anchor_batch).to(device)
            positive_tensor = torch.stack(positive_batch).to(device)
            negative_tensor = torch.stack(negative_batch).to(device)

            anchor_embedding = self.encoder(anchor_tensor)
            positive_embedding = self.encoder(positive_tensor)
            negative_embedding = self.encoder(negative_tensor)
            loss = self.contrastive_learner.contrastive_loss(
                anchor_embedding,
                positive_embedding,
                negative_embedding,
            )

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(self.encoder.parameters())
                + list(self.contrastive_learner.parameters()),
                max_norm=1.0,
            )
            self.optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1
        return epoch_loss / num_batches

    def predict_dispatching_action(self, current_state: Dict) -> Dict[str, str]:
        """Score actions with the untrained policy head and greedily match them."""
        if "recent_history" in current_state:
            history = current_state["recent_history"][-self.window_length :]
        else:
            history = [current_state]

        # Left-padding supplies a fixed sequence length when only one state is given.
        encoded = [self.encode_terminal_state(state) for state in history]
        while len(encoded) < self.window_length:
            encoded.insert(0, torch.zeros(self.fixed_feature_size, dtype=torch.float32))
        state_sequence = torch.stack(encoded).unsqueeze(0).to(device)

        self.encoder.eval()
        self.policy.eval()
        with torch.no_grad():
            state_representation = self.encoder(state_sequence)
            action_logits = self.policy(state_representation)
            action_scores = torch.softmax(action_logits, dim=-1).squeeze(0).cpu().numpy()

        return self.generate_assignments_from_scores(current_state, action_scores)

    def generate_assignments_from_scores(
        self, state: Dict, scores: np.ndarray
    ) -> Dict[str, str]:
        """Greedily assign distinct represented trucks using fixed action slots."""
        assignments: Dict[str, str] = {}
        available_trucks = [
            f"T{i + 1}"
            for i in range(self.max_trucks)
            if f"T{i + 1}" in state["trucks"]
        ]
        containers = state["containers"][: self.max_containers]
        score_matrix = scores[: self.max_containers * self.max_trucks].reshape(
            self.max_containers, self.max_trucks
        )

        # Action slot (container i, truck j) is always i * max_trucks + j.
        # Keeping this layout fixed makes scores interpretable across states.
        for container_index, container in enumerate(containers):
            if not available_trucks:
                break
            available_indices = [
                int(truck_id[1:]) - 1 for truck_id in available_trucks
            ]
            row = score_matrix[container_index, available_indices]
            best_position = int(np.argmax(row))
            truck_id = available_trucks.pop(best_position)
            assignments[container["id"]] = truck_id
        return assignments


print("Self-supervised dispatcher is defined.")

### How the learning components fit together

A **state** is one snapshot of trucks, containers, and system indicators. A **sequence window** is an ordered set of five such snapshots. The LSTM processes the snapshots in order, so its hidden state can carry information from earlier time steps. Self-attention then lets each encoded time step use information from the other steps in the same window. Mean pooling converts the resulting sequence into one vector, which is convenient for comparing whole windows.

The contrastive task needs no manually assigned class labels: the sampling rule supplies a positive window and a negative window for each anchor. The loss trains the encoder and projection head to distinguish those two candidates. This is self-supervision in the limited sense that the training signal is constructed from the observations' positions in the history.

The policy head has a different role. Its output contains one score for each fixed container-truck action slot, but the contrastive objective contains no policy parameters and therefore cannot teach the head which dispatch action is useful. A trained representation and an untrained action head must not be conflated. The implementation keeps these components separate so that the objective's actual training signal is visible.

In [ ]:
def generate_synthetic_historical_data(num_periods: int = 100) -> List[Dict]:
    """Generate reproducible random snapshots for a demonstration dataset."""
    historical_data = []
    locations = {
        "berth": (0, 0),
        "storage": (5, 5),
        "rail": (10, 0),
        "gate": (0, 10),
        "yard": (5, 0),
    }
    location_names = list(locations)

    for period in range(num_periods):
        num_trucks = np.random.randint(3, 6)
        num_containers = np.random.randint(2, 8)

        trucks = {}
        for i in range(num_trucks):
            truck_id = f"T{i + 1}"
            location = locations[random.choice(location_names)]
            trucks[truck_id] = {
                "location_x": location[0],
                "location_y": location[1],
                "available_time": int(np.random.randint(0, 5)),
                "current_load": int(np.random.randint(0, 2)),
            }

        containers = []
        for i in range(num_containers):
            origin_name = random.choice(location_names)
            dest_name = random.choice(location_names)
            while dest_name == origin_name:
                dest_name = random.choice(location_names)
            containers.append(
                {
                    "id": f"C{period * 10 + i + 1}",
                    "origin_x": locations[origin_name][0],
                    "origin_y": locations[origin_name][1],
                    "dest_x": locations[dest_name][0],
                    "dest_y": locations[dest_name][1],
                    "priority": int(np.random.randint(5, 20)),
                    "earliest_pickup": period,
                    "latest_delivery": period + int(np.random.randint(3, 10)),
                }
            )

        average_urgency = float(np.mean([c["priority"] for c in containers]))
        truck_utilization = sum(
            truck["current_load"] for truck in trucks.values()
        ) / len(trucks)
        historical_data.append(
            {
                "period": period,
                "trucks": trucks,
                "containers": containers,
                "average_urgency": average_urgency,
                "truck_utilization": truck_utilization,
            }
        )
    return historical_data


print("Generating synthetic historical data...")
historical_data = generate_synthetic_historical_data(num_periods=100)
print(f"Generated {len(historical_data)} synthetic periods.")
print(
    "Average containers per period: "
    f"{np.mean([len(state['containers']) for state in historical_data]):.1f}"
)
print(
    "Average trucks per period: "
    f"{np.mean([len(state['trucks']) for state in historical_data]):.1f}"
)

sample_state = historical_data[0]
print("\nSample state structure:")
print(f"- Period: {sample_state['period']}")
print(f"- Trucks: {len(sample_state['trucks'])}")
print(f"- Containers: {len(sample_state['containers'])}")
print(f"- Average urgency: {sample_state['average_urgency']:.2f}")
print(f"- Truck utilization: {sample_state['truck_utilization']:.2f}")

### Training objective and interpretation

For each sampled triple, the model compares an anchor window with one positive and one negative window. The positive is the next overlapping five-period window; the negative is selected from windows whose start periods differ from the anchor start by more than ten. The windows are encoded separately, then the projection head maps each embedding to a unit vector. Unit normalization makes the dot product a cosine similarity.

The cross-entropy target is class zero because the first logit in each row is the anchor-positive similarity. Minimizing the loss increases the positive logit relative to the negative logit. Gradient clipping limits the norm of the encoder and projection-head gradients before the Adam optimizer updates those parameters. There is no policy loss, reward, or action label in this training loop; the policy head remains untrained.

In [ ]:
config = {
    "state_dim": 43,
    "num_actions": 50,
    "num_trucks": 5,
    "max_containers": 5,
    "window_length": 5,
}

dispatcher = SelfSupervisedDispatcher(config)
print(f"Configuration: {config}")
print(
    "Encoder parameters: "
    f"{sum(parameter.numel() for parameter in dispatcher.encoder.parameters()):,}"
)
print(f"Training history periods: {len(historical_data)}")

num_epochs = 10
print(f"\nStarting contrastive training for {num_epochs} epochs...")
start_time = time.time()

for epoch in range(num_epochs):
    epoch_loss = dispatcher.train_epoch(historical_data)
    dispatcher.training_losses.append(epoch_loss)

    if len(dispatcher.training_losses) > 1:
        previous_loss = dispatcher.training_losses[-2]
        if not np.isclose(previous_loss, 0.0):
            change_rate = (previous_loss - epoch_loss) / abs(previous_loss)
        else:
            change_rate = float("nan")
        dispatcher.loss_change_rates.append(change_rate)

    print(f"Epoch {epoch + 1}/{num_epochs}: contrastive loss = {epoch_loss:.4f}")

training_time = time.time() - start_time
print(f"\nContrastive training completed in {training_time:.2f} seconds.")

if len(dispatcher.training_losses) > 1:
    initial_loss = dispatcher.training_losses[0]
    final_loss = dispatcher.training_losses[-1]
    if not np.isclose(initial_loss, 0.0):
        loss_change_percent = (initial_loss - final_loss) / abs(initial_loss) * 100
        print(f"Change in training loss from first to last epoch: {loss_change_percent:.1f}%")
    else:
        print("The first-epoch loss is approximately zero; a percentage change is not reported.")

A decreasing training loss means that the model is fitting the sampled contrastive comparisons more successfully. It does not establish that the embeddings preserve operationally useful information, that the model generalizes to real terminals, or that dispatch decisions improve. The loss-change rate is a change in the optimization objective, not an independent representation-quality metric. A stronger assessment would require an independently defined downstream task and appropriate validation data; neither is supplied by this synthetic example.

In [ ]:
print("\n" + "=" * 60)
print("ACTION-SCORING DEMONSTRATION")
print("=" * 60)

# The policy head is untrained; this state is used only to demonstrate its interface.
test_data = generate_synthetic_historical_data(num_periods=10)
test_state = test_data[0]
print("Test-state characteristics:")
print(f"- Trucks: {len(test_state['trucks'])}")
print(f"- Containers: {len(test_state['containers'])}")
print(f"- Average urgency: {test_state['average_urgency']:.2f}")

start_inference = time.time()
assignments = dispatcher.predict_dispatching_action(test_state)
inference_time = time.time() - start_inference
print(f"\nScoring and matching time: {inference_time:.4f} seconds")
print(f"Assignments made: {len(assignments)}")

locations = {
    "berth": (0, 0),
    "storage": (5, 5),
    "rail": (10, 0),
    "gate": (0, 10),
    "yard": (5, 0),
}

def nearest_location_name(x: float, y: float) -> str:
    """Name the closest listed terminal location using Manhattan distance."""
    return min(
        locations,
        key=lambda name: abs(x - locations[name][0]) + abs(y - locations[name][1]),
    )


if assignments:
    print("\nDemonstration assignments from untrained action scores:")
    print(f"{'Container':<12} {'Truck':<8} {'Priority':<10} {'Route':<20}")
    print("-" * 55)
    for container_id, truck_id in assignments.items():
        container = next(
            item for item in test_state["containers"] if item["id"] == container_id
        )
        origin_name = nearest_location_name(
            container["origin_x"], container["origin_y"]
        )
        destination_name = nearest_location_name(
            container["dest_x"], container["dest_y"]
        )
        route = f"{origin_name} to {destination_name}"
        print(
            f"{container_id:<12} {truck_id:<8} "
            f"{container['priority']:<10} {route:<20}"
        )
else:
    print("No assignments were generated.")

print(
    "\nInterpretation: the matching procedure enforces at most one assignment per "
    "represented truck, but the untrained scores have no learned operational meaning."
)

### Descriptive comparison with a reactive baseline

The reactive baseline sorts the current containers by priority and assigns them to trucks in dictionary order. Its utility is the sum of assigned-container priorities. The same utility is calculated for assignments produced by the untrained action head. This comparison is intentionally descriptive: it does not measure travel time, lateness, throughput, or an optimized dispatch objective, and the two assignment rules do not receive equivalent training. Utility and coverage should therefore not be interpreted as evidence of predictive learning or operational superiority.

In [ ]:
def reactive_assignment(state: Dict) -> Dict[str, str]:
    """Assign highest-priority jobs to trucks in their listed order."""
    assignments = {}
    available_trucks = [
        f"T{i + 1}"
        for i in range(dispatcher.max_trucks)
        if f"T{i + 1}" in state["trucks"]
    ]
    sorted_containers = sorted(
        state["containers"][: dispatcher.max_containers],
        key=lambda container: container["priority"],
        reverse=True,
    )
    for container, truck_id in zip(sorted_containers, available_trucks):
        assignments[container["id"]] = truck_id
    return assignments


def assignment_metrics(state: Dict, assignments: Dict[str, str]) -> Dict[str, float]:
    """Calculate assignment count, priority sum, and container coverage."""
    priority_by_id = {
        container["id"]: container["priority"]
        for container in state["containers"]
    }
    utility = sum(priority_by_id[container_id] for container_id in assignments)
    container_count = len(state["containers"])
    coverage = len(assignments) / container_count if container_count else 0.0
    return {
        "assignments": len(assignments),
        "utility": utility,
        "coverage": coverage,
    }


test_cases = generate_synthetic_historical_data(num_periods=20)
predictive_performance = []
reactive_performance = []

for case_number, test_case in enumerate(test_cases, start=1):
    predictive_assignments = dispatcher.predict_dispatching_action(test_case)
    reactive_assignments = reactive_assignment(test_case)

    predictive_metrics = assignment_metrics(test_case, predictive_assignments)
    reactive_metrics = assignment_metrics(test_case, reactive_assignments)
    predictive_performance.append({"case": case_number, **predictive_metrics})
    reactive_performance.append({"case": case_number, **reactive_metrics})

pred_df = pd.DataFrame(predictive_performance)
reac_df = pd.DataFrame(reactive_performance)
summary_stats = pd.DataFrame(
    {
        "Untrained policy head": [
            pred_df["assignments"].mean(),
            pred_df["utility"].mean(),
            pred_df["coverage"].mean(),
        ],
        "Reactive priority baseline": [
            reac_df["assignments"].mean(),
            reac_df["utility"].mean(),
            reac_df["coverage"].mean(),
        ],
    },
    index=["Average assignments", "Average priority sum", "Average coverage"],
)

print("Descriptive performance summary:")
print(summary_stats.round(3))

def relative_change_percent(candidate: float, baseline: float) -> Optional[float]:
    """Return relative change when the baseline is nonzero."""
    if np.isclose(baseline, 0.0):
        return None
    return (candidate - baseline) / abs(baseline) * 100


print("\nRelative changes for descriptive context:")
for metric, label in [
    ("assignments", "Assignment count"),
    ("utility", "Priority sum"),
    ("coverage", "Container coverage"),
]:
    change = relative_change_percent(
        pred_df[metric].mean(), reac_df[metric].mean()
    )
    if change is None:
        print(f"- {label}: relative change is undefined because the baseline mean is zero.")
    else:
        print(f"- {label}: {change:+.1f}%")

print(
    "These metrics describe outputs on synthetic cases; they do not establish "
    "the quality of the learned representation or a trained dispatch policy."
)

### Visualizing the training signal and demonstration metrics

The first panel plots the contrastive training loss. The second plots its epoch-to-epoch relative change, which is an optimization diagnostic rather than a representation-quality score. The remaining panels compare assignment count, priority-sum utility, and coverage on the synthetic evaluation cases. Each metric has its own scale and interpretation; bars should not be compared across different metrics as though they shared units.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

epochs = np.arange(1, len(dispatcher.training_losses) + 1)
ax1.plot(
    epochs,
    dispatcher.training_losses,
    color="blue",
    linewidth=2,
    marker="o",
    label="Training loss",
)
ax1.set_title("Contrastive Training Loss", fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("InfoNCE-style cross-entropy loss")
ax1.grid(True, alpha=0.3)
ax1.legend()

change_epochs = np.arange(2, len(dispatcher.training_losses) + 1)
if dispatcher.loss_change_rates:
    ax2.plot(
        change_epochs,
        dispatcher.loss_change_rates,
        color="darkorange",
        linewidth=2,
        marker="s",
        label="Relative loss reduction",
    )
ax2.axhline(0.0, color="black", linewidth=1, alpha=0.6)
ax2.set_title("Epoch-to-Epoch Loss Change", fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Relative loss reduction")
ax2.grid(True, alpha=0.3)
ax2.legend()

methods = ["Untrained policy head", "Reactive priority baseline"]
metric_names = ["Assignments", "Priority sum", "Coverage"]
metric_values = [
    [pred_df["assignments"].mean(), reac_df["assignments"].mean()],
    [pred_df["utility"].mean(), reac_df["utility"].mean()],
    [pred_df["coverage"].mean(), reac_df["coverage"].mean()],
]
x = np.arange(len(metric_names))
width = 0.36
ax3.bar(x - width / 2, [values[0] for values in metric_values], width, label=methods[0])
ax3.bar(x + width / 2, [values[1] for values in metric_values], width, label=methods[1])
ax3.set_title("Mean Synthetic-Case Metrics", fontweight="bold")
ax3.set_xlabel("Metric")
ax3.set_ylabel("Metric value (units differ by metric)")
ax3.set_xticks(x)
ax3.set_xticklabels(metric_names)
ax3.legend()
ax3.grid(True, alpha=0.3, axis="y")

case_numbers = pred_df["case"].to_numpy()
ax4.plot(
    case_numbers,
    pred_df["assignments"],
    color="green",
    linewidth=2,
    marker="o",
    label="Untrained policy head",
)
ax4.plot(
    case_numbers,
    reac_df["assignments"],
    color="red",
    linewidth=2,
    marker="s",
    label="Reactive priority baseline",
)
ax4.set_title("Assignments by Synthetic Test Case", fontweight="bold")
ax4.set_xlabel("Test case")
ax4.set_ylabel("Number of assignments")
ax4.set_xticks(case_numbers)
ax4.grid(True, alpha=0.3)
ax4.legend()

fig.suptitle("Training Diagnostics and Descriptive Assignment Evaluation", fontsize=14)
fig.tight_layout()
plt.show()

### Representation and action-space limits

The feature encoder has five truck slots and five container slots, while the policy head has 50 action outputs. Only the first 25 outputs are used by the fixed five-by-five truck-container score layout; the remaining logits are unused. States containing more than five trucks or more than five containers are not fully represented by this configuration. To keep the scale experiment within the defined model, it varies the number of represented trucks and containers up to five each. Results from this small synthetic experiment are descriptive and do not establish generalization to larger terminals.

In [ ]:
scale_scenarios = [
    {"name": "Small scale", "num_trucks": 3, "num_containers": 3},
    {"name": "Medium scale", "num_trucks": 5, "num_containers": 5},
    {"name": "Reduced truck availability", "num_trucks": 2, "num_containers": 5},
]
scale_results = []

for scenario in scale_scenarios:
    scenario_data = generate_synthetic_historical_data(num_periods=5)

    for state in scenario_data:
        truck_ids = list(state["trucks"])
        for truck_id in truck_ids[scenario["num_trucks"] :]:
            del state["trucks"][truck_id]

        state["containers"] = state["containers"][: scenario["num_containers"]]
        state["average_urgency"] = float(
            np.mean([container["priority"] for container in state["containers"]])
        )
        state["truck_utilization"] = sum(
            truck["current_load"] for truck in state["trucks"].values()
        ) / len(state["trucks"])

    assignment_counts = []
    priority_sums = []
    for state in scenario_data:
        scenario_assignments = dispatcher.predict_dispatching_action(state)
        metrics = assignment_metrics(state, scenario_assignments)
        assignment_counts.append(metrics["assignments"])
        priority_sums.append(metrics["utility"])

    mean_assignments = float(np.mean(assignment_counts))
    mean_priority_sum = float(np.mean(priority_sums))
    scale_results.append(
        {
            "Scenario": scenario["name"],
            "Trucks": scenario["num_trucks"],
            "Containers": scenario["num_containers"],
            "Mean assignments": mean_assignments,
            "Mean priority sum": mean_priority_sum,
            "Assignment rate": mean_assignments / scenario["num_containers"],
            "Priority sum per assignment": (
                mean_priority_sum / mean_assignments if mean_assignments else 0.0
            ),
        }
    )

scale_df = pd.DataFrame(scale_results)
print("Descriptive results within the configured feature and action limits:")
print(scale_df.round(3))

assignment_rate_std = scale_df["Assignment rate"].std(ddof=0)
priority_per_assignment_std = scale_df["Priority sum per assignment"].std(ddof=0)
print("\nAcross-scenario descriptive variability:")
print(f"- Standard deviation of assignment rate: {assignment_rate_std:.3f}")
print(
    "- Standard deviation of priority sum per assignment: "
    f"{priority_per_assignment_std:.3f}"
)
print(
    "These statistics describe the generated cases and an untrained policy head; "
    "they are not a validation of generalization."
)

## Relationship to the surrounding tiers

This tier introduces a data-driven representation-learning component alongside the dispatching approaches covered elsewhere in the problem. Its focus is not to replace a mathematical formulation, an auction-based heuristic, or a population-based search method. Instead, it illustrates how historical observations can be converted into training pairs without manually supplied class labels.

The contrastive objective learns to separate the sampled positive and negative windows. Whether that separation captures useful terminal structure depends on the data and pair-sampling assumptions. A temporal encoder does not automatically forecast demand, and an action-scoring network does not become a dispatch policy merely because it receives an encoder embedding. The policy head requires an appropriate training signal and evaluation against operational objectives before its decisions can be treated as learned dispatch decisions.

### Potential strengths and limitations

**Potential strengths of the representation-learning component**

- It constructs a training signal from the ordering of historical observations, without manually assigned class labels.
- The LSTM and attention layers can represent temporal relationships within the supplied windows.
- The same encoder architecture can be studied with larger or more representative histories, subject to appropriate validation.

**Limitations of this demonstration**

- The synthetic observations are randomly generated and do not reproduce calibrated terminal dynamics.
- Temporal adjacency and distance are proxies for positive and negative similarity, not verified semantic labels.
- The action head is untrained, so its scores have no learned operational interpretation.
- The fixed feature and action layouts limit the represented problem size, and the greedy assignment does not optimize travel, deadlines, or other operational costs.
- Training loss alone is not a measure of downstream representation quality, safety, or dispatch performance.

### When this approach may be worth studying

Self-supervised temporal representation learning may be worth investigating when substantial, time-ordered operational data is available and manually labeled examples are limited. A practical study would need to validate the sampling assumptions, define an operational downstream objective, train the action component with a defensible signal, and evaluate performance on representative held-out data. Those steps are outside the scope of this synthetic demonstration.

## Summary

This notebook implements a temporal encoder with an LSTM and multi-head attention, constructs adjacent-positive and temporally distant-negative windows, and trains the encoder and projection head using an InfoNCE-style contrastive objective. It also defines a policy head and demonstrates how its action scores can be converted into one-to-one greedy assignments.

The key distinction is between **representation training** and **policy training**. The contrastive loss updates the encoder and projection head, but it does not update the policy head. Consequently, the assignments and synthetic-case comparisons are demonstrations of the scoring and matching interface, not evidence that the notebook has learned a dispatch policy or achieved operational performance improvements.

The model's fixed state vector contains 43 features: four features for each of five truck slots, four for each of five container slots, and three system-level indicators. A five-state window provides the temporal input to the encoder. Missing truck and container slots are zero-padded, and the fixed action layout uses the first 25 logits for the five-by-five container-truck combinations.

### Further study

Natural extensions include testing the representation on a validated downstream task, using operationally justified positive and negative pair definitions, incorporating additional measured state variables, and evaluating on held-out terminal data. Any policy-training objective, additional operational constraints, or larger action space would require a clearly specified modeling choice and evaluation design; they are not implemented here.